# YOLO 데이터셋 준비

- YOLO Classification용 데이터셋 생성
- 원본 스크립트: `aws_icon_yolo_cls_prepare_and_train.sh`

## 개요
AWS Icon Classification (YOLO-CLS) Dataset Builder

### 용도
- Stage1 (coarse 19 클래스) / Stage2 (fine 64 클래스) 아이콘 분류용 YOLO 데이터셋 생성
- YOLOv8 분류 모델 학습에 바로 사용할 수 있는 디렉터리 구조를 만든다.

### 전제 조건
프로젝트 루트 구조:
```
./dataset/icons/
  ├─ images/
  ├─ taxonomy_coarse.csv
  ├─ taxonomy_fine.csv
  ├─ labels_fine.csv
  ├─ train_fine.csv
  ├─ val_fine.csv
  └─ test_fine.csv
```

### 사용법
```bash
# fine (64 클래스) 아이콘 분류 데이터셋 생성
./aws_icon_yolo_cls_prepare_and_train.sh fine ./dataset/icons

# coarse (19 클래스) 아이콘 분류 데이터셋 생성
./aws_icon_yolo_cls_prepare_and_train.sh coarse ./dataset/icons
```

### 학습 예시
```bash
yolo classify train \
  data=./dataset/icons/yolo_cls_fine \
  model=weights/yolov8n-cls.pt \
  epochs=50 imgsz=256
```

### 요구 패키지
- Python >= 3.8
- pandas

In [97]:
from __future__ import annotations

import os, argparse, shutil
from pathlib import Path

import pandas as pd

# ----- Helpers -----
def parse_args() -> argparse.Namespace:
    """노트북에서 argv 충돌 없이 MODE/DATA_DIR을 설정한다."""

    parser = argparse.ArgumentParser(description="YOLO-CLS 데이터셋 준비")
    parser.add_argument("--mode", choices=["fine", "coarse"], default=os.environ.get("MODE", "fine"))
    parser.add_argument("--data-dir", default=os.environ.get("DATA_DIR", "./dataset/icons"))
    return parser.parse_args(args=[])


def ensure_dir(path: Path) -> Path:
    if not path.is_dir():
        raise FileNotFoundError(f"디렉터리가 없습니다: {path}")
    return path


# ---- Config -----
args = parse_args()
MODE: str = args.mode
DATA_DIR: Path = ensure_dir(Path(args.data_dir).resolve())
IMG_DIR: Path = ensure_dir((DATA_DIR / "images").resolve())
train_path: Path = DATA_DIR / "train_fine.csv"
val_path: Path = DATA_DIR / "val_fine.csv"
test_path: Path = DATA_DIR / "test_fine.csv"

os.environ["MODE"] = MODE
os.environ["DATA_DIR"] = str(DATA_DIR)

print(f"[INFO] MODE     = {MODE}")
print(f"[INFO] DATA_DIR = {DATA_DIR}")

[INFO] MODE     = fine
[INFO] DATA_DIR = /home/wsm/workspace/hit-archlens-project/dataset/icons


In [98]:
# config 체크만 수행 (값 정의는 앞 셀)
if not IMG_DIR.is_dir():
    raise FileNotFoundError(f"images dir not found: {IMG_DIR}")

if not train_path.exists() or not val_path.exists() or not test_path.exists():
    raise FileNotFoundError("train/val/test_fine.csv 중 하나 이상이 없습니다.")

### 1. taxonomy 로드 및 클래스 ID 매핑 생성

In [99]:
from typing import Tuple
from IPython.display import display


def load_taxonomy(mode: str) -> tuple[pd.DataFrame, str]:
    """taxonomy CSV를 읽어 정렬된 클래스 테이블을 반환한다."""

    label_col = "canonical_service_name" if mode == "fine" else "coarse_class"
    csv_path = DATA_DIR / f"taxonomy_{mode}.csv"
    if not csv_path.exists():
        raise FileNotFoundError(csv_path)

    classes = (
        pd.read_csv(csv_path)[label_col]
        .dropna()
        .pipe(lambda s: s.str.strip() if s.dtype == object else s)
        .drop_duplicates()
        .sort_values()
        .reset_index(drop=True)
        .to_frame(name=label_col)
    )
    classes["class_id"] = classes.index
    return classes, label_col


classes, label_col = load_taxonomy(MODE)
name_to_id = classes.set_index(label_col)["class_id"].to_dict()
id_to_name = classes.set_index("class_id")[label_col].to_dict()

print(f"[INFO] {MODE} 클래스 개수: {len(classes)}")
print("[INFO] 클래스 매핑 상위 10개:")
display(classes.head(10))

[INFO] fine 클래스 개수: 64
[INFO] 클래스 매핑 상위 10개:


,canonical_service_name,class_id
0,amazon api gateway,0
1,amazon athena,1
2,amazon aurora,2
3,amazon braket,3
4,amazon chime,4
5,amazon cloudwatch,5
6,amazon comprehend,6
7,amazon connect,7
8,amazon documentdb,8
9,amazon dynamodb,9


### 2. split CSV 로드

In [100]:
def load_split(path: Path) -> pd.DataFrame:
    df = pd.read_csv(path)
    if label_col not in df.columns:
        raise KeyError(f"{label_col} not in {path.name}")
    return df

df_train = load_split(train_path)
df_val = load_split(val_path)
df_test = load_split(test_path)

print(f"[INFO] rows - train: {len(df_train)}, val: {len(df_val)}, test: {len(df_test)}")

[INFO] rows - train: 447, val: 96, test: 96


### 3. 출력 디렉터리 구조 준비 

In [101]:
OUT_ROOT = DATA_DIR / f"yolo_cls_{MODE}"
if OUT_ROOT.exists():
    print(f"[WARN] {OUT_ROOT} 이미 존재합니다. 내용을 모두 지웁니다.")
    shutil.rmtree(OUT_ROOT)

for split in ["train", "val", "test"]:
    split_dir = OUT_ROOT / split
    split_dir.mkdir(parents=True, exist_ok=True)

print(f"[OK] 출력 루트 디렉터리: {OUT_ROOT}")

[WARN] /home/wsm/workspace/hit-archlens-project/dataset/icons/yolo_cls_fine 이미 존재합니다. 내용을 모두 지웁니다.
[OK] 출력 루트 디렉터리: /home/wsm/workspace/hit-archlens-project/dataset/icons/yolo_cls_fine


### 4. 이미지 심볼릭 링크 생성

In [102]:
def build_split(df: pd.DataFrame, split_name: str):
    split_dir = OUT_ROOT / split_name
    n_missing = 0
    total = len(df)

    for idx, row in df.iterrows():
        label_name = row[label_col]
        class_id = name_to_id.get(label_name)
        if class_id is None:
            raise KeyError(f"'{label_name}' 에 해당하는 class_id가 없습니다. MODE={MODE}")

        src_rel = row["file_path"]
        src_path = (IMG_DIR / src_rel).resolve()  # 절대경로로 고정해 링크 깨짐 방지
        if not src_path.is_file():
            n_missing += 1
            if n_missing <= 10:
                print(f"[WARN] missing image: {src_path}")
            continue

        class_dir = split_dir / str(class_id)
        class_dir.mkdir(parents=True, exist_ok=True)

        # 파일명 충돌 방지를 위해 split, 인덱스 prefix 사용
        dst_name = f"{split_name}_{idx}_{src_path.name}"
        dst_path = class_dir / dst_name

        try:
            # 심볼릭 링크 사용 (공간 절약). 필요시 shutil.copy2로 대체 가능.
            if not dst_path.exists():
                os.symlink(src_path, dst_path)
        except OSError:
            # 일부 환경에서 symlink가 안 될 수도 있으므로 fallback: copy
            shutil.copy2(src_path, dst_path)

    print(f"[OK] {split_name}: {total - n_missing}개 링크 생성, {n_missing}개 미존재")

build_split(df_train, "train")
build_split(df_val, "val")
build_split(df_test, "test")

print("[DONE] YOLO 분류용 데이터셋 생성 완료.")
print(f"       경로: {OUT_ROOT}")

[OK] train: 447개 링크 생성, 0개 미존재
[OK] val: 96개 링크 생성, 0개 미존재
[OK] test: 96개 링크 생성, 0개 미존재
[DONE] YOLO 분류용 데이터셋 생성 완료.
       경로: /home/wsm/workspace/hit-archlens-project/dataset/icons/yolo_cls_fine


In [103]:
!ls dataset/icons/yolo_cls_fine/train | wc -l
!ls dataset/icons/yolo_cls_fine/val | wc -l
!ls dataset/icons/yolo_cls_fine/test | wc -l

64
64
64


In [104]:
expected = set(classes["class_id"])
def coverage(df, name):
    present = set(df[label_col].map(name_to_id))
    missing = sorted(expected - present)
    print(f"{name}: {len(present)}/{len(expected)} classes present")
    if missing:
        print("missing class_ids:", missing)
coverage(df_train, "train")
coverage(df_val, "val")
coverage(df_test, "test")

train: 64/64 classes present
val: 64/64 classes present
test: 64/64 classes present


## 참고: YOLO 학습 명령어

데이터셋 생성 후 다음 명령어로 학습할 수 있습니다:

```bash
# fine 또는 coarse 분류 모델 학습
yolo classify train \
  data=./dataset/icons/yolo_cls_fine \
  model=weights/yolov8n-cls.pt \
  epochs=50 imgsz=256

# 평가
yolo classify val \
  model=runs/classify/train/weights/best.pt \
  data=./dataset/icons/yolo_cls_fine
```

In [105]:
# 데이터셋 무결성 검증 (11번 노트북 실행 전 최종 체크)
from collections import Counter

OUT_ROOT = DATA_DIR / f"yolo_cls_{MODE}"
expected_classes = len(classes)


def check_split(split: str) -> dict:
    split_dir = OUT_ROOT / split
    if not split_dir.is_dir():
        return {"ok": False, "error": f"missing dir: {split_dir}"}

    class_dirs = [p for p in split_dir.iterdir() if p.is_dir()]
    broken = 0
    total = 0
    for cls_dir in class_dirs:
        for f in cls_dir.iterdir():
            total += 1
            if f.is_symlink() and not f.exists():
                broken += 1
    return {
        "ok": True,
        "classes": len(class_dirs),
        "files": total,
        "broken": broken,
    }

reports = {}
for split, df in [("train", df_train), ("val", df_val), ("test", df_test)]:
    stat = check_split(split)
    stat["csv_rows"] = len(df)
    reports[split] = stat
    status = "OK" if stat.get("ok") else "NG"
    print(
        f"[{status}] {split}: classes={stat.get('classes')} / {expected_classes}, "
        f"files={stat.get('files')} (csv_rows={stat['csv_rows']}), "
        f"broken_links={stat.get('broken')}"
    )
    if stat.get("broken"):
        print("    -> broken symlinks detected; 원본 이미지/경로 확인 필요")

print("\n검증 기준:")
print("- classes가 기대값(64 fine / 19 coarse)과 같아야 합니다.")
print("- files 수가 csv_rows 이상이어야 하며 broken_links는 0이어야 합니다.")
print(f"- 데이터 루트: {OUT_ROOT}")


[OK] train: classes=64 / 64, files=447 (csv_rows=447), broken_links=0
[OK] val: classes=64 / 64, files=96 (csv_rows=96), broken_links=0
[OK] test: classes=64 / 64, files=96 (csv_rows=96), broken_links=0

검증 기준:
- classes가 기대값(64 fine / 19 coarse)과 같아야 합니다.
- files 수가 csv_rows 이상이어야 하며 broken_links는 0이어야 합니다.
- 데이터 루트: /home/wsm/workspace/hit-archlens-project/dataset/icons/yolo_cls_fine
